# VirTues integration

[VirTues](https://www.nature.com/articles/s41586-026-10884-y#) is a foundation model for highly multiplexed tissue images. The model takes as inputs the marker intensities for selected marker image channels, and applies a vision transformer architecture to learn an embedding of the tissue image. This embedding can be used for downstream tasks such as cell type annotation, clustering, or tissue annotation. VirTues can also be used for segmentation. 

In `spatialproteomics`, we can run VirTues directly on the `xarray` dataset, with support for `SpatialData` objects planned. This tutorial illustrates how you can use VirTues to obtain a segmentation, or to obtain a cellular embedding which you can use, e.g., for clustering cells by their learned representations.

What you find in this tutorial:

- [What you need](#what-you-need)
- [Running VirTues on an `xarray` object](#running-virtues-on-an-xarray-object)
  - [Setup and data loading](#setup-and-data-loading)
  - [Marker mapping](#marker-mapping)
  - [Preprocessing](#preprocessing)
  - [The VirTues function: two modes](#the-virtues-function-two-modes)
  - [Segment](#segment)
  - [Embed cells](#embed-cells)
- [Step-by-step workflow with `xarray`](#step-by-step-workflow-with-xarray)
  - [Validating the marker-to-protein mapping](#validating-the-marker-to-protein-mapping)
  - [Preprocessing](#step-by-step-preprocessing)
  - [Loading the VirTues model](#loading-the-virtues-model)
  - [Running inference: `mode='segment'`](#running-inference-segment)
  - [Running inference: `mode='embed_cells'`](#running-inference-embed-cells)

<a id="what-you-need"></a>

## What you need 

If you want to follow along with this tutorial, you can download the data [here](https://oc.embl.de/index.php/s/XzEa9po1tjiDzzJ).

> **Note**
>
> What you also need to run this (more explanation below):
>
> - a mapping of your marker channels to the corresponding UniProt protein IDs
> - access to an NVIDIA GPU with CUDA
> - a Hugging Face token if requested when downloading the model
> - ideally, dataset-level marker means and standard deviations for standardisation

Install VirTues and its dependencies following the [VirTues installation instructions](https://github.com/bunnelab/virtues#installation), and make the `virtues` package importable in your notebook environment. 

The additional dataset-level statistics used here are read from `../../data/global_mean_std_after_clip_log1p.csv`; these should contain `marker_name`, `mean`, and `std` columns computed after quantile clipping and `log1p`.

<a id="running-virtues-on-an-xarray-object"></a>

## Running VirTues on an `xarray` object

<a id="setup-and-data-loading"></a>

### Setup and data loading

In [1]:
import spatialproteomics as sp
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
ds = xr.open_zarr("../../data/LN_11_1.zarr")
ds

<xarray.Dataset> Size: 7GB
Dimensions:            (cells: 16871, cells_2: 16871, channels: 56, y: 3000,
                        x: 3000, la_features: 4, labels: 19, la_props: 2,
                        neighborhoods: 6, nh_props: 2, features: 21)
Coordinates:
  * cells              (cells) int64 135kB 1 2 3 4 5 ... 16868 16869 16870 16871
  * cells_2            (cells_2) int64 135kB 1 2 3 4 ... 16868 16869 16870 16871
  * channels           (channels) <U11 2kB 'DAPI' 'Helios' ... 'CD79a' 'Ki-67'
  * features           (features) object 168B 'BCL-2_binarized' ... 'inter_la...
  * la_features        (la_features) object 32B 'labels_0' ... 'labels_3'
  * la_props           (la_props) <U6 48B '_color' '_name'
  * labels             (labels) int64 152B 1 2 3 4 5 6 7 ... 14 15 16 17 18 19
  * neighborhoods      (neighborhoods) int64 48B 1 2 3 4 5 6
  * nh_props           (nh_props) <U6 48B '_color' '_name'
  * x                  (x) int64 24kB 0 1 2 3 4 5 ... 2995 2996 2997 2998 2999
  * y                  (y) int64 24kB 0 1 2 3 4 5 ... 2995 2996 2997 2998 2999
Data variables:
    _adjacency_matrix  (cells, cells_2) int64 2GB ...
    _image             (channels, y, x) float64 4GB ...
    _image_raw         (channels, y, x) uint8 504MB ...
    _intensity         (cells, channels) float64 8MB ...
    _la_layers         (cells, la_features) object 540kB ...
    _la_properties     (labels, la_props) object 304B ...
    _neighborhoods     (cells, labels) float64 3MB ...
    _nh_properties     (neighborhoods, nh_props) <U14 672B ...
    _obs               (cells, features) float64 3MB ...
    _segmentation      (y, x) int64 72MB ...

In [ ]:
celltype_colors = {
    "B cell": "#5799d1",
    "T cell": "#ebc850",
    "Myeloid cell": "#de6866",
    "Dendritic cell": "#4cbcbd",
    "Macrophage": "#bb7cb4",
    "Stromal cell": "#62b346",
    "Endothelial cell": "#bf997d",
}

<a id="marker-mapping"></a>

### Marker mapping

We also need to provide the model with the biological meaning of the different image channels. To do so, the model needs to incorporate information about each of the used markers by using a learned representation of the marker proteins, a numerical embedding matrix that captures markers' biological identity and relationships. For this, each marker has to be mapped to a UniProt protein ID. The corresponding protein sequence is downloaded from UniProt and embedded with a protein language model ([ESM](https://github.com/facebookresearch/esm)). The resulting marker embedding matrix is then used by the VirTues encoder together with the preprocessed image.

In the `spatialproteomics` workflow, the user has to supply an explicit marker-protein ID mapping as a `pandas` `DataFrame`. Default expected column names are `marker_name` and `protein_id`; other names can be passed to `ds.tl.virtues()` as arguments. 

> **Note**
>
> In VirTues, the UniProt protein ID is used to download the respective protein's sequence as a `fasta` file. This is then embedded with a protein language model (specifically [ESM](https://github.com/facebookresearch/esm)). In the original `virtues` package code, these `fasta` files and the corresponding embeddings are stored on disk as separate files for each marker/protein. In the `ds.tl.virtues()` function in `spatialproteomics`, this is instead computed on the fly and the final embedding matrix is added to the `xarray` object directly (under `_virtues_marker_embeddings`).

> **Note**
>
> **DAPI:** DAPI is a nuclear DNA stain and as such, it has no direct protein target. Here, it is included as pseudo-Histone-H3 with UniProt P68431, applying the VirTues authors' convention for non-protein nuclear markers such as Hoechst/Ir191/Ir193. The authors explicitly describe mapping these nuclear markers to the histone H3 amino acid sequence in [this Methods paragraph](https://www.nature.com/articles/s41586-026-10884-y#:~:text=To%20facilitate%20cell%20segmentation%2C%20when%20histone%20H3).

Let's define the marker-to-protein ID mapping:

In [3]:
protein_mapping_dict = {
    "DAPI": "P68431",
    "PAX5": "Q02548",
    "CD3": "P07766",
    "CD11b": "P11215",
    "CD11c": "P20702",
    "CD68": "P34810",
    "CD90": "P04216",
    "Podoplanin": "Q86YL7",
    "CD31": "P16284",
    "CD34": "P28906",
    "Ki-67": "P46013",
    "CD8": "P01732",
    "CD4": "P01730",
}
protein_mapping = pd.DataFrame(
    protein_mapping_dict.items(), columns=["marker_name", "protein_id"]
)
protein_mapping

,marker_name,protein_id
0,DAPI,P68431
1,PAX5,Q02548
2,CD3,P07766
3,CD11b,P11215
4,CD11c,P20702
5,CD68,P34810
6,CD90,P04216
7,Podoplanin,Q86YL7
8,CD31,P16284
9,CD34,P28906


The function selects the image channels listed in `protein_mapping` internally, in mapping order.
Channels not included in the mapping are not used for VirTues, but remain in the returned dataset.
Mapping entries whose marker names are absent from the image are dropped with a warning.

<a id="preprocessing"></a>

### Preprocessing 

For preprocessing, we need to standardise the image, i.e., subtract for each marker channel and each image the mean of that marker across the entire dataset, and divide by the standard deviation of that marker across the entire dataset 

> **Note**
>
> This means that we need dataset-level marker means and standard deviations, calculated in advance on all quantile-clipped, `log1p`-transformed images in the dataset. As a fallback, the implementation uses the image-level mean and standard deviation after clipping and `log1p`. However, this can lead to markers with zero std which will raise an error and have to be excluded manually.

Here, we supply the dataset-level mean and standard deviation directly: 

In [4]:
global_stats = pd.read_csv('../../data/global_mean_std_after_clip_log1p.csv')
global_stats.head()

,channel_id,marker_name,protein_id,mean,std,n_pixels
0,0,DAPI,P68431,2.710769,1.167353,2250000000
1,1,Helios,Q9UKS7,0.033555,0.295935,2250000000
2,2,CD10,P08473,1.480504,1.391294,2250000000
3,3,TCF7/TCF1,P36402,0.307866,0.844906,2250000000
4,4,PD-L1,Q9NZQ7,1.325507,1.531718,2250000000


The `spatialproteomics` wrapper expects `mean` and `std` as `pandas.Series` indexed by marker name:

In [ ]:
global_stats_indexed = global_stats.set_index("marker_name")
mean = global_stats_indexed["mean"]
std = global_stats_indexed["std"]

<a id="the-virtues-function-two-modes"></a>

### The VirTues function: two modes

Now, we can call `ds.tl.virtues()`. It has two modes, the `'segment'` mode and the `'embed_cells'` mode. In both modes, it will first complete a series of preprocessing steps, described in more detail [below](#step-by-step-workflow-with-xarray).

In the `'segment'` mode, `ds.tl.virtues()` will then call the segmentation head to output a segmentation mask based on the extracted embeddings. By default, the segmentation mask is stored in the `xarray` object, alongside the marker embeddings. If desired, patch embeddings can be stored in addition by setting `store_patch_embeddings=True` (default: `False`). The function returns the updated `xarray` with the new fields. 

In the `'embed_cells'` mode, the function will aggregate the spatial token embeddings to a cell-level embedding. For this, a segmentation has to be already present in the object. For each cell, a cell-level embedding is calculated by aggregating all patch tokens using a pixel-weighted overlap. I.e., we calculate a weighted average of the embeddings of all patch tokens that overlap with the cell (based on the segmentation), where the weights are determined by the number of pixels in the cell that overlap with each token. This mirrors the definition in VirTues' [`compute_cell_tokens()`](https://github.com/bunnelab/virtues/blob/main/virtues/utils/cell_tokens.py). The cell embeddings are stored under `_virtues_cell_embeddings` by default, alongside the marker embeddings. If desired, patch-level embeddings can be stored in addition by setting `store_patch_embeddings=True`.

<a id="segment"></a>

### Segment

First, we use VirTues for segmentation. For this, we create a copy containing only the raw image and its channel and spatial coordinates. The function stores the segmentation under the default key `_segmentation`. 

If you want to use it in addition to an already existing segmentation to compare them, you can do this by specifying `segmentation_key_added="_virtues_segmentation"`. 

However, please note that your cell ID table etc and obs if you have them will remain linked to the canonical segmentation, and the cell IDs in the VirTues segmentation will not correspond to those, so be careful.

In [ ]:
# Start from the raw image only, using the standard image key.
ds_for_segmentation = ds[["_image_raw"]].rename({"_image_raw": "_image"}).copy()
sp_obj_virtues = ds_for_segmentation.tl.virtues(
    protein_mapping=protein_mapping,
    mean=mean,
    std=std,
)
sp_obj_virtues

Optional arguments you can set:

- `image_key`: image layer used as input; default: `'_image'`. Our minimal datasets store a copy of the raw image under this default key.
- `marker_col` and `protein_id_col`: mapping column names; defaults: `'marker_name'` and `'protein_id'`.
- `model_name`: pretrained model; default: `'virtues-sp32'` (currently the only supported model).
- `cache_dir`: Hugging Face model cache directory; default: `None`, using the Hugging Face default cache.
- `virtues_path`: local VirTues checkout to add to the import path; default: `None`.
- `quantile`: per-image, per-channel clipping quantile; default: `0.99`.
- `mean` and `std`: dataset-level statistics indexed by marker name; defaults: `None`, using image-level statistics.
- `gaussian_blur`: apply Gaussian blur after standardisation; default: `True`.
- `gaussian_blur_sigma`: blur sigma in pixels; default: `1.0`.
- `store_patch_embeddings`: store spatial tokens and their crop coordinates; default: `False`.
- `tile_size`: crop size in pixels; default: `128`.
- `segment_overlap`: segmentation overlap parameter in pixels; default: `32`. The total crop overlap is `2 * segment_overlap` for images up to 1500 pixels on their longest side, or `2 * (segment_overlap + 20)` for larger images.
- `cell_token_stride`: crop stride in pixels for `mode="embed_cells"`; default: `42`.
- `batch_size`: crops processed together; default: `None`, resolved to `4` for segmentation or `32` for cell embeddings.
- `segmentation_key_added`: segmentation output key; default: `'_segmentation'`. An existing canonical segmentation cannot be replaced, even with `overwrite=True`.
- `cell_type_prediction_key_added`: optional pixel-wise class prediction key; default: `None` (not stored).
- `cell_embedding_key_added`: cell embedding output key; default: `'_virtues_cell_embeddings'`.
- `marker_embedding_key_added`: marker embedding output key; default: `'_virtues_marker_embeddings'`.
- `patch_embedding_key_added`: patch embedding output key; default: `'_virtues_patch_embeddings'`.
- `patch_coord_key_added`: crop coordinate output key; default: `'_virtues_patch_coords'`.
- `overwrite`: replace existing custom outputs and recompute marker embeddings; default: `False`.
- `device`: execution device; default: `'cuda'`, currently the only supported value.

In [ ]:
example_crop = sp_obj_virtues.pp["DAPI"].pp[1000:1300, 1000:1300].pl.colorize("gray")

fig, ax = plt.subplots(1, 2, figsize=(14, 7))
_ = example_crop.pl.show(ax=ax[0])
_ = example_crop.pl.show(render_segmentation=True, ax=ax[1])
ax[0].set_title("DAPI")
ax[1].set_title("DAPI with VirTues segmentation")
plt.tight_layout()

<a id="embed-cells"></a>

### Embed cells

Now, for demonstration purposes, we will run the function again in the `'embed_cells'` mode to get cell-level embeddings. This will re-run the spatial token extraction, but not run the segmentation. 

> **Note**
>
> If you want to get both a segmentation and cell-level embeddings, you should run `ds.tl.virtues()` in the `'segment'` mode with `store_patch_embeddings=True`. Then, you can use the function `tl.virtues_aggregate_patch_embeddings()` to aggregate the patch tokens into cell-level embeddings using the computed segmentation:
>
> ```python
> sp_obj_with_tokens = ds_for_segmentation.tl.virtues(
>     protein_mapping=protein_mapping,
>     mean=mean,
>     std=std,
>     store_patch_embeddings=True,
> )
> sp_obj_with_tokens = sp_obj_with_tokens.tl.virtues_aggregate_patch_embeddings()
> ```

You can also store pixel-wise cell type predictions by setting `cell_type_prediction_key_added="_virtues_cell_type_prediction"`. These predictions are not automatically assigned to cells; the step-by-step example below shows the explicit projection.

Storing patch embeddings during segmentation requires the [adapted VirTues checkout](https://github.com/maren-ha/virtues/tree/spatialproteomics-patch-embeddings) that supports `segment_tissue(..., return_patch_embeddings=True)`. Use an importable installation of that checkout, or pass its path as `virtues_path`. 

Before the embedding call, we create a fresh copy from the original dataset, keeping the raw image, segmentation, and cell-type labels. Unlike segmentation mode, `embed_cells` requires an existing segmentation to define the cells. The examples below extract embeddings for the original segmentation so that we can compare the clusters with the existing cell-type labels. 

In [ ]:
# Start again from the original cells and labels, without prior analysis outputs.
ds_for_embedding = ds[
    ["_image_raw", "_segmentation", "_obs", "_la_layers", "_la_properties"]
].rename({"_image_raw": "_image"}).copy()
ds_for_embedding = ds_for_embedding.sel(features=["centroid-0", "centroid-1", "_labels"])
sp_obj_embed = ds_for_embedding.tl.virtues(
    mode="embed_cells",
    protein_mapping=protein_mapping,
    mean=mean,
    std=std,
)
sp_obj_embed["_virtues_cell_embeddings"]

We can now use these cell embeddings for all sorts of downstream tasks. As an example illustration, we will run K-means clustering with five clusters here. 

We cluster the cell embeddings directly, using `KMeans(n_clusters=5, random_state=0, n_init=10)` as an illustrative setting. Cells without a finite embedding are excluded from clustering. The reference labels are the original broad cell types (`labels_0`); these are used only for comparison, not to construct the clusters.

In [ ]:
from sklearn.cluster import KMeans

embeddings = sp_obj_embed["_virtues_cell_embeddings"].to_pandas()
embeddings = embeddings.loc[np.isfinite(embeddings).all(axis=1)]
cluster_labels = pd.Series(
    KMeans(n_clusters=5, random_state=0, n_init=10).fit_predict(embeddings),
    index=embeddings.index,
    name="cluster",
)
cell_types = ds_for_embedding["_la_layers"].sel(la_features="labels_0").to_pandas()
cluster_table = cluster_labels.to_frame().join(cell_types.rename("cell_type"))
cluster_table.head()

In [ ]:
# Add cluster labels to a separate plotting object, preserving the original labels.
plot_labels = pd.Series("Unlabeled", index=sp_obj_embed.cells.values)
plot_labels.loc[cluster_table.index] = "Cluster " + cluster_table["cluster"].astype(str)
ds_clusters = sp_obj_embed.pp.drop_layers(keep=["_image", "_segmentation", "_obs"])
ds_clusters = ds_clusters.la.add_labels(plot_labels.to_dict())
ds_labels = ds.la.set_label_level("labels_0", ignore_neighborhoods=True)
ds_labels = ds_labels.la.set_label_colors(celltype_colors.keys(), celltype_colors.values())

fig, ax = plt.subplots(1, 2, figsize=(14, 7))
_ = ds_labels.pp["DAPI"].pl.colorize("gray").pl.show(render_labels=True, ax=ax[0])
_ = ds_clusters.pp["DAPI"].pl.colorize("gray").pl.show(render_labels=True, ax=ax[1])
ax[0].set_title("Original cell types")
ax[1].set_title("K-means clusters")
plt.tight_layout()

In [ ]:
composition = pd.crosstab(cluster_table["cluster"], cluster_table["cell_type"], normalize="index")
composition.plot.bar(stacked=True, color=celltype_colors, figsize=(10, 5))
plt.xlabel("K-means cluster")
plt.ylabel("Fraction of cells")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
composition

Adjusted Rand index and normalised mutual information compare the clusters with the
existing broad cell-type labels. Larger values indicate closer agreement; these labels
are a reference annotation, not an independent ground truth.

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

pd.Series({
    "Adjusted Rand index": adjusted_rand_score(cluster_table["cell_type"], cluster_table["cluster"]),
    "Normalised mutual information": normalized_mutual_info_score(
        cluster_table["cell_type"], cluster_table["cluster"]
    ),
})

<a id="step-by-step-workflow-with-xarray"></a>

## Step-by-step workflow with `xarray`

The above is all that is needed to run VirTues through `spatialproteomics`. In the following, we take the function apart and run one step at a time, to illustrate how the workflow runs internally. If you just want to get a segmentation or cell embeddings, just use the calls shown above. 

<a id="validating-the-marker-to-protein-mapping"></a>

### Validating the marker-to-protein mapping

Validate the marker-to-protein ID mapping and select the markers present in the image, preserving the mapping order.

In [ ]:
from spatialproteomics.tl import virtues_utils
from spatialproteomics.tl import virtues_backends

mapping = virtues_utils.validate_and_normalize_protein_mapping(
    ds_for_segmentation.channels.values.tolist(),
    protein_mapping,
    marker_col="marker_name",
    protein_id_col="protein_id",
    model_name="virtues-sp32",
)
mapping

<a id="step-by-step-preprocessing"></a>

### Preprocessing

Preprocess the image: this is customisable, but the default follows the VirTues approach
([Methods](https://www.nature.com/articles/s41586-026-10884-y#Sec7)):

- compute a pre-specified quantile `q` (`virtues` default: 0.99) for each marker channel over the entire image
- clip each channel to `[0, its quantile value]`
- apply a `log1p` transformation
- standardise the image using dataset-level marker means and standard deviations
- optionally, apply a Gaussian blur with prespecified parameters (default: enabled, with sigma 1.0 pixels)

In [ ]:
selected_image = ds_for_segmentation["_image"].sel(channels=mapping["marker_name"].tolist()).values
image, preprocessing_attrs = virtues_utils.preprocess_for_virtues(
    selected_image, mapping, mean=mean, std=std
)
image.shape, preprocessing_attrs

<a id="loading-the-virtues-model"></a>

### Loading the VirTues model 

For running VirTues, the pre-trained model has to be loaded from the [huggingface checkpoint](https://huggingface.co/bunnelab/virtues) (default: the `virtues-sp32` model trained on data from diverse technologies incl. CODEX, see [here](https://github.com/bunnelab/virtues/tree/main#models) for an overview of their available models).

This is handled in the function `spatialproteomics.tl.virtues_utils.load_virtues_model()`, which is called internally by `ds.tl.virtues()`. The checkpoints are downloaded to the default Hugging Face cache, or to the directory passed as `cache_dir`. If authentication is required, log in with `hf auth login` before running the notebook, or configure `HF_TOKEN` in your environment. 

You need a GPU to run model inference; CPU-only runs are currently not supported by VirTues (because it [relies on FlashAttention](https://github.com/bunnelab/virtues/issues/11)). 

First compute ESM marker embeddings from the mapped protein sequences. Then load the VirTues backbone and, for segmentation, the segmentation head from Hugging Face or the local cache.

In [ ]:
marker_embeddings = virtues_utils._compute_virtues_marker_embeddings(
    mapping["protein_id"].tolist()
)
sp_obj_steps = virtues_backends.add_marker_embeddings(
    ds_for_segmentation, marker_embeddings, mapping, "_virtues_marker_embeddings"
)
virtues_model, segmentation_model = virtues_utils.load_virtues_model(
    marker_embeddings, model_name="virtues-sp32", load_segmentation_head=True
)

<a id="running-inference-segment"></a>

### Running inference: `mode='segment'`

Tile the image into 128 x 128 pixel crops by default. With `segment_overlap=32`, the nominal
total crop overlap is 64 pixels for images up to 1500 pixels on their longest side, or 104 pixels
for larger images. Each crop is divided into 8 x 8 pixel patches; the model extracts
`patch_summary_tokens`, the spatial tokens aggregated across marker channels.

The pretrained U-Net segmentation head uses these representations to produce a segmentation
and pixel-wise cell type predictions. Here, we also request the patch embeddings and their crop
coordinates so that they can be aggregated into cell embeddings later.

In [ ]:
segmentation_result = virtues_utils._run_virtues_segmentation(
    image,
    segmentation_model,
    tile_size=128,
    segment_overlap=32,
    batch_size=4,
    return_cell_type_prediction=True,
    return_patch_embeddings=True,
)

In [ ]:
sp_obj_steps = virtues_backends.add_virtues_segmentation_outputs(
    sp_obj_steps,
    segmentation_result["segmentation"],
    segmentation_result["cell_type_prediction"],
    segmentation_key_added="_segmentation",
    cell_type_prediction_key_added="_virtues_cell_type_prediction",
    attrs={**segmentation_result["segmentation_metadata"], **preprocessing_attrs,
           "model_name": "virtues-sp32", "image_key": "_image"},
)
sp_obj_steps = virtues_backends.add_patch_embeddings(
    sp_obj_steps,
    segmentation_result["patch_embeddings"],
    segmentation_result["patch_coords_yx"],
    patch_embedding_key_added="_virtues_patch_embeddings",
    patch_coord_key_added="_virtues_patch_coords",
    attrs={**segmentation_result["patch_metadata"], **preprocessing_attrs,
           "model_name": "virtues-sp32", "image_key": "_image"},
)
sp_obj_steps

<a id="running-inference-embed-cells"></a>

### Running inference: `mode='embed_cells'`

In cell embedding mode, the image is also divided into 128 x 128 pixel crops, with a default
crop stride of 42 pixels. For each original cell, the 8 x 8 pixel patch embeddings are averaged
using their pixel overlap with that cell as weights. This runs the backbone without the
segmentation head, using the original segmentation in `ds_for_embedding`.

In [ ]:
# This workflow uses the original segmentation, as in the high-level example.
embedding_result = virtues_utils._run_virtues_cell_embedding(
    image,
    ds_for_embedding["_segmentation"].values,
    virtues_model,
    tile_size=128,
    cell_token_stride=42,
    batch_size=32,
)
sp_obj_embed_steps = virtues_backends.add_virtues_embedding_outputs(
    ds_for_embedding,
    embedding_result["cell_ids"],
    embedding_result["cell_embeddings"],
    key_added="_virtues_cell_embeddings",
    attrs={**embedding_result["cell_embedding_metadata"], **preprocessing_attrs,
           "model_name": "virtues-sp32", "image_key": "_image",
           "segmentation_key_used": "_segmentation"},
)
sp_obj_embed_steps["_virtues_cell_embeddings"]

The segmentation result also contains pixel-wise cell type predictions when requested.
The class dictionary is stored in the prediction layer's attributes. The wrapper suppresses
`None` and `Unknown` before taking the per-pixel argmax. To obtain one label per cell,
`project_virtues_cell_types_to_existing_segmentation()` takes the most frequent class among
the pixels in each canonical `_segmentation` cell; it adds a column to `_la_layers` without
changing the active labels.

In [ ]:
sp_obj_steps["_virtues_cell_type_prediction"].attrs["cell_type_mapping"]

In [ ]:
sp_obj_steps = sp_obj_steps.tl.project_virtues_cell_types_to_existing_segmentation()
sp_obj_steps.pp.get_layer_as_df("_la_layers")["_virtues_cell_type"].value_counts()

To annotate the original cells instead, add the same pixel-wise prediction layer to the
original dataset and project onto its canonical segmentation. The active original labels
remain available for comparison.

In [ ]:
ds_projected = ds_for_embedding.pp.add_layer(
    sp_obj_steps["_virtues_cell_type_prediction"].values,
    key_added="_virtues_cell_type_prediction",
)
ds_projected = ds_projected.tl.project_virtues_cell_types_to_existing_segmentation()
ds_projected.pp.get_layer_as_df("_la_layers")[["labels_0", "_virtues_cell_type"]].head()

The stored segmentation-run tokens can also be aggregated into embeddings for the new VirTues cells, without rerunning inference. 

In [ ]:
sp_obj_steps = sp_obj_steps.tl.virtues_aggregate_patch_embeddings()
sp_obj_steps["_virtues_cell_embeddings"]